In [1]:
!pip install peft bitsandbytes accelerate nltk --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.7 MB/s eta 0:00:00


In [2]:
import re, random, nltk, torch, gc, time
from collections import Counter
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

nltk.download("words", quiet=True)
from nltk.corpus import words as nltk_words
ENGLISH_WORDS = set(w.upper() for w in nltk_words.words() if 3 <= len(w) <= 7 and w.isalpha())
print(f"📚 {len(ENGLISH_WORDS):,} English words loaded")


📚 57,440 English words loaded


In [3]:
GAME_GROUPS = {"ABT":["BAT","TAB"],"ACT":["ACT","CAT"],"AET":["ATE","EAT","TEA","ETA"],"APT":["APT","TAP","PAT"],"ART":["ART","RAT","TAR"],"ADM":["DAM","MAD"],"AMP":["AMP","MAP"],"ANP":["NAP","PAN"],"ANT":["ANT","TAN"],"APS":["SAP","SPA","ASP"],"AEL":["ALE","LEA"],"DEN":["DEN","END"],"GNU":["GNU","GUN","NUG"],"NOW":["NOW","OWN","WON"],"OPT":["OPT","TOP","POT"],"ORT":["ROT","TOR"],"DGO":["DOG","GOD"],"LOW":["OWL","LOW"],"ENT":["NET","TEN"],"INP":["PIN","NIP"],"IPS":["SIP","PSI"],"ACRS":["CARS","SCAR","ARCS"],"ACST":["CATS","CAST","ACTS","SCAT"],"AELP":["PALE","LEAP","PLEA","PEAL"],"AELR":["REAL","EARL"],"AELS":["SALE","SEAL","ALES"],"AELT":["LATE","TALE","TEAL"],"AEMN":["NAME","MEAN","MANE","AMEN"],"AENR":["NEAR","EARN"],"AEPS":["PEAS","APES"],"AEPT":["TAPE","PEAT","PATE"],"AERW":["WEAR","WARE"],"ALPS":["SLAP","LAPS","ALPS","PALS"],"ANPS":["SNAP","SPAN","NAPS","PANS"],"ARST":["STAR","RATS","ARTS","TARS"],"DEIS":["SIDE","DIES","IDES"],"EILS":["LIES","ISLE"],"EIST":["SITE","TIES"],"ENOT":["NOTE","TONE"],"EORS":["ROSE","ORES","SORE"],"OPST":["STOP","POST","TOPS","SPOT","OPTS","POTS"],"ORST":["SORT","ROTS"],"ILNO":["LOIN","LION"],"ADEL":["DALE","DEAL","LEAD"],"AMST":["MAST","MATS","TAMS"],"EILV":["VILE","LIVE","EVIL","VEIL"],"GINS":["SING","SIGN","GINS"],"ADEM":["MADE","DAME","MEAD"],"ADER":["DARE","READ","DEAR"],"AEGM":["GAME","MAGE"],"AELM":["MALE","LAME","MEAL"],"AMOR":["ROAM","MORA"],"AELPS":["LEAPS","PALES","SEPAL","PEALS","PLEAS"],"AELRT":["LATER","ALTER","ALERT"],"AELST":["STEAL","TALES","STALE","LEAST","SLATE"],"AENRS":["SNARE","EARNS","NEARS","SANER"],"AEPRS":["SPARE","SPEAR","PARSE","PEARS","REAPS"],"AEGRS":["GEARS","RAGES","SAGER"],"AILNS":["NAILS","SNAIL","SLAIN"],"AINRT":["TRAIN","INTRA"],"DEIRS":["RIDES","SIRED","DRIES"],"AELNP":["PENAL","PANEL","PLANE"],"EILPS":["PILES","PLIES","SPIEL"],"EINPS":["PINES","SPINE","SNIPE"],"EINRS":["REINS","RINSE","SIREN","RISEN"],"EINST":["INSET","STEIN","TINES"],"EORST":["STORE","ROTES","TORES"],"ERSTW":["STREW","WREST"],"AILRT":["TRAIL","TRIAL"],"ADELS":["DEALS","LEADS","DALES"],"AELNS":["LANES","LEANS"],"DEIST":["DIETS","EDITS","TIDES","SITED"],"ADEMS":["MEADS","DAMES"],"AELRST":["STELAR","ALERTS","ALTERS","SLATER"],"AEINRS":["ARISEN","SARNIE"],"ADEIRS":["RAISED","DARIES"],"AEGNRS":["RANGES","ANGERS"],"DEISTU":["SUITED","DUTIES"],"EINORS":["SENIOR","IRONES"],"AEGLNS":["ANGLES","GLEANS"],"AEINST":["TISANE","INSEAT"],"AEPRST":["PASTER","REPAST"],"ACENRS":["CANERS","CRANES","NACRES"],"AEGNRT":["GARNET","ARGENT"],"AELRSV":["SALVER","VELARS","LAVERS"],"DEGINS":["DESIGN","SIGNED","SINGED"],"AGINST":["GIANTS","SATING"],"DEINRS":["DINERS","RINSED"],"AEMNST":["STAMEN","AMENTS","MANTES"],"EILNST":["LISTEN","SILENT","TINSEL","ENLIST"],"EINPRS":["SNIPER","RIPENS"],"AEGINR":["EARING","GAINER","REGAIN"],"AEILNRS":["ALINERS","NAILERS"],"AEINRST":["NASTIER","RETAINS","STAINER"],"ADEINRS":["SARDINE","RANDIES"],"AEGINRS":["SEARING","ERASING","REGAINS"],"AELPRST":["PLASTER","PSALTER","STAPLER"],"AEILNST":["SALTINE","ELASTIN","ENTAILS"],"ACEIRST":["RACIEST","STEARIC","CRISTAE"],"AEGINST":["SEATING","TEASING","INGESTA"],"ADEGNRS":["GARDENS","GANDERS","DANGERS"],"AEINPRS":["RAPINES","PANIERS"],"ACELPRS":["CLASPER","PARCELS","SCALPER"],"EINORST":["STONIER","ORIENTS"],"AELMNOT":["OMENTAL","TELAMON"],"ADEINST":["INSTEAD","DETAINS","SAINTED"]}

SYSTEM_MSG = "You are an anagram solver. Given scrambled letters, find ALL valid English words that use every letter exactly once. Return words as uppercase CSV. No explanation."

def make_prompts(n=500):
    random.seed(42)
    keys = [k for k, v in GAME_GROUPS.items() if len(set(v)) >= 2]
    prompts = []
    for _ in range(n):
        key = random.choice(keys)
        letters = list(key)
        random.shuffle(letters)
        prompts.append({"text": f"Unscramble: {' '.join(letters)}", "letters": key})
    return prompts

PROMPTS = make_prompts(500)
print(f"📊 {len(PROMPTS)} prompts")


📊 500 prompts


In [4]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", quantization_config=bnb_config,
    device_map="auto", torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

lora_config = LoraConfig(r=32, lora_alpha=32, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

model.print_trainable_parameters()
print("✅ Model loaded (HuggingFace + PEFT, no Unsloth)")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

trainable params: 20,185,088 || all params: 771,817,472 || trainable%: 2.6153
✅ Model loaded (HuggingFace + PEFT, no Unsloth)


In [5]:
def compute_reward(response_text, letter_key):
    text = response_text.strip()
    if "</think>" in text:
        text = text.split("</think>")[-1].strip()
    text = text.split("\n")[0].strip()
    words = [w.strip().upper() for w in text.split(",") if w.strip()]
    if not words or not any(w.isalpha() for w in words):
        return -2.0
    letter_counter = Counter(letter_key.upper())
    total = 0.5 if all(w.isalpha() for w in words) else 0.0
    seen = set()
    for word in words:
        if not word.isalpha() or word in seen:
            total -= 0.5; continue
        seen.add(word)
        if Counter(word) == letter_counter:
            total += 2.0 if word in ENGLISH_WORDS else -0.5
        else:
            total -= 1.0
    return total
print(f"🧪 'BAT, TAB' → {compute_reward('BAT, TAB', 'ABT')}")


🧪 'BAT, TAB' → 4.5


In [9]:
from torch.utils.data import Dataset, DataLoader

class AnagramDataset(Dataset):
    def __init__(self, tokenizer, n=2000):
        random.seed(42)
        self.examples = []
        keys = list(GAME_GROUPS.keys())
        for _ in range(n):
            key = random.choice(keys)
            letters = list(key)
            random.shuffle(letters)
            prompt = f"Unscramble: {' '.join(letters)}"
            answer = ", ".join(GAME_GROUPS[key])
            messages = [
                {"role": "system", "content": SYSTEM_MSG},
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": answer}
            ]
            text = tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)
            enc = tokenizer(text, truncation=True, max_length=256, padding="max_length", return_tensors="pt")
            self.examples.append({
                "input_ids": enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels": enc["input_ids"].squeeze().clone()
            })
    def __len__(self): return len(self.examples)
    def __getitem__(self, i): return self.examples[i]

dataset = AnagramDataset(tokenizer, n=2000)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

model.train()
sft_optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4, weight_decay=0.01)
print("📖 SFT Warm-up: Teaching the model anagram format...")
print("=" * 50)
sft_losses = []
for epoch in range(2):
    for i, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["labels"].to(model.device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        sft_optimizer.step()
        sft_optimizer.zero_grad()
        sft_losses.append(loss.item())
        if (i+1) % 50 == 0:
            print(f"  Epoch {epoch+1} | Batch {i+1}/{len(loader)} | Loss: {sum(sft_losses[-50:])/50:.3f}")
    gc.collect(); torch.cuda.empty_cache()

# Quick test
model.eval()
test_msgs = [{"role":"system","content":SYSTEM_MSG},{"role":"user","content":"Unscramble: T B A"}]
test_input = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
test_enc = tokenizer(test_input, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**test_enc, max_new_tokens=64, do_sample=False, pad_token_id=tokenizer.pad_token_id)
print(f"\n✅ SFT Done! Test: 'T B A' → {tokenizer.decode(out[0][test_enc['input_ids'].shape[1]:], skip_special_tokens=True)}")


📖 SFT Warm-up: Teaching the model anagram format...
  Epoch 1 | Batch 50/500 | Loss: 0.594
  Epoch 1 | Batch 100/500 | Loss: 0.084
  Epoch 1 | Batch 150/500 | Loss: 0.074
  Epoch 1 | Batch 200/500 | Loss: 0.063
  Epoch 1 | Batch 250/500 | Loss: 0.058
  Epoch 1 | Batch 300/500 | Loss: 0.055
  Epoch 1 | Batch 350/500 | Loss: 0.051
  Epoch 1 | Batch 400/500 | Loss: 0.050
  Epoch 1 | Batch 450/500 | Loss: 0.048
  Epoch 1 | Batch 500/500 | Loss: 0.049
  Epoch 2 | Batch 50/500 | Loss: 0.046
  Epoch 2 | Batch 100/500 | Loss: 0.045
  Epoch 2 | Batch 150/500 | Loss: 0.045
  Epoch 2 | Batch 200/500 | Loss: 0.046
  Epoch 2 | Batch 250/500 | Loss: 0.045
  Epoch 2 | Batch 300/500 | Loss: 0.044
  Epoch 2 | Batch 350/500 | Loss: 0.044
  Epoch 2 | Batch 400/500 | Loss: 0.045
  Epoch 2 | Batch 450/500 | Loss: 0.043
  Epoch 2 | Batch 500/500 | Loss: 0.045


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



✅ SFT Done! Test: 'T B A' → BAT, TAB


In [10]:
@torch.no_grad()
def generate_responses(model, tokenizer, prompt_text, n=4):
    messages = [{"role":"system","content":SYSTEM_MSG},{"role":"user","content":prompt_text}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False,
        add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    responses = []
    for _ in range(n):
        out = model.generate(**inputs, max_new_tokens=64, do_sample=True,
            temperature=0.8, top_p=0.95, pad_token_id=tokenizer.pad_token_id)
        responses.append(tokenizer.decode(out[0][input_len:], skip_special_tokens=True))
    return responses, input_text

def grpo_step(model, tokenizer, optimizer, prompt_data):
    model.eval()
    responses, full_prompt = generate_responses(model, tokenizer, prompt_data["text"])
    rewards = [compute_reward(r, prompt_data["letters"]) for r in responses]
    rewards_t = torch.tensor(rewards, dtype=torch.float32)
    advantages = (rewards_t - rewards_t.mean()) / (rewards_t.std() + 1e-8)
    if rewards_t.std() < 1e-6:
        return rewards_t.mean().item(), responses, rewards
    model.train()
    optimizer.zero_grad()
    total_loss = torch.tensor(0.0, device=model.device, requires_grad=True)
    for response, adv in zip(responses, advantages):
        if adv.item() <= 0: continue
        full_text = full_prompt + response + tokenizer.eos_token
        enc = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=256)
        input_ids = enc["input_ids"].to(model.device)
        prompt_len = tokenizer(full_prompt, return_tensors="pt")["input_ids"].shape[1]
        logits = model(input_ids=input_ids, attention_mask=enc["attention_mask"].to(model.device)).logits
        log_probs = F.log_softmax(logits[:, prompt_len-1:-1, :], dim=-1)
        token_lp = torch.gather(log_probs, 2, input_ids[:, prompt_len:].unsqueeze(-1)).squeeze(-1)
        total_loss = total_loss + (-adv.to(model.device) * token_lp.mean())
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return rewards_t.mean().item(), responses, rewards

NUM_STEPS = 150
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-6, weight_decay=0.01)
print("🚀 GRPO: 150 steps × 4 responses (SFT → GRPO pipeline)")
print("=" * 50)
reward_history, start_time = [], time.time()
for step in range(1, NUM_STEPS + 1):
    prompt = random.choice(PROMPTS)
    mean_r, responses, rewards = grpo_step(model, tokenizer, optimizer, prompt)
    reward_history.append(mean_r)
    if step % 5 == 0:
        avg = sum(reward_history[-5:]) / 5
        eta = (time.time()-start_time)/step*(NUM_STEPS-step)/60
        best_idx = rewards.index(max(rewards))
        print(f"Step {step:>3}/{NUM_STEPS} | R:{mean_r:>+5.1f} | Avg:{avg:>+5.1f} | ETA:{eta:.0f}m")
        print(f"  {prompt['text']} → {responses[best_idx][:60]}")
    if step % 50 == 0: gc.collect(); torch.cuda.empty_cache()
print(f"\n✅ Done! ({(time.time()-start_time)/60:.0f}min)")


🚀 GRPO: 150 steps × 4 responses (SFT → GRPO pipeline)
Step   5/150 | R: +4.5 | Avg: +4.1 | ETA:12m
  Unscramble: T A N → ANT, TAN
Step  10/150 | R: +4.5 | Avg: +1.4 | ETA:11m
  Unscramble: T O R → ROT, TOR
Step  15/150 | R: +3.5 | Avg: +3.6 | ETA:11m
  Unscramble: A E L T S R → STELAR, ALERTS, ALTERS, SLATER
Step  20/150 | R: +4.0 | Avg: +4.5 | ETA:10m
  Unscramble: S P N I E → PINES, SPINE, SNIPE
Step  25/150 | R: +2.0 | Avg: +4.2 | ETA:10m
  Unscramble: S E T I → SITE, TIES
Step  30/150 | R: +4.5 | Avg: +3.2 | ETA:9m
  Unscramble: I N L O → LOIN, LION
Step  35/150 | R: +4.5 | Avg: +2.8 | ETA:9m
  Unscramble: P S I → SIP, PSI
Step  40/150 | R: -1.0 | Avg: +2.2 | ETA:8m
  Unscramble: S C R E N A → CANERS, CRANES, NACRES
Step  45/150 | R: -1.1 | Avg: +4.1 | ETA:8m
  Unscramble: E I T D S U → SUITED, DUTIES
Step  50/150 | R: -1.0 | Avg: +4.0 | ETA:8m
  Unscramble: A D N R E G S → GARDENS, GANDERS, DANGERS
Step  55/150 | R: +2.0 | Avg: +3.4 | ETA:7m
  Unscramble: N E L S A G → ANGLES, GLE